# AI Lab - Practical Question No. 15
## BFS 2D Grid Maze Solver (Implicit Graph Search)

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Problem Solving
- **Core AI Concept**: Grid Search, Successor Function, Implicit Graph Exploration
- **Python Concepts Tested**: 2D array indexing, coordinate tuples, `collections.deque`, set hashing
- **Expected Algorithm**: Breadth-First Search (BFS) on Implicit Graphs
- **Recommended Libraries**: `collections`
- **Complexity**: Time Complexity: $O(N \times M)$ | Space Complexity: $O(N \times M)$
- **Constraints / Conditions**: Grid can be non-square ($N \ne M$)
- **Sample Input**: `[[0, 1, 0], [0, 0, 0], [1, 1, 0]]`
- **Sample Output**: `Shortest steps: 4`
- **Optional Challenge**: Output a string of moves: `"D, R, R, D"`
- **University Practical Importance**: High

---

### Problem Statement
Design a Python program that uses **Breadth-First Search (BFS)** to solve a 2D grid maze. The maze is represented as a 2D list where `0` is an open path and `1` is a wall. The agent can move **Up**, **Down**, **Left**, and **Right** (orthogonal transitions). Find the shortest path from the top-left corner `(0, 0)` to the bottom-right corner `(N-1, M-1)`.

### Task Requirements
1. **Parse the 2D Grid**: Support 2D lists, non-square dimensions, and ASCII/JSON formatted inputs.
2. **Dynamic Successor Generation**: Boundary bounds checking ($0 \le r < N, 0 \le c < M$) and obstacle collision checks (`cell == 0`).
3. **Track Visited Coordinates**: Track `(row, col)` tuples to prevent duplicate insertions and cycle loops.
4. **Return Shortest Path Length**: Integer representing the minimum steps/actions to reach the goal.
5. **Optional Challenge**: Reconstruct the chronological string of moves: `"D, R, R, D"`.

### 1. Theoretical Foundations & Algorithmic Principles

#### Implicit vs. Explicit Graphs
- **Explicit Graphs**: All vertices $V$ and edges $E$ are stored in advance in memory (e.g. adjacency lists or adjacency matrices). While suitable for static road maps or social networks, explicit graphs require $O(V + E)$ pre-allocated space.
- **Implicit Graphs**: Vertices and edges are **not** pre-allocated. Instead, they are generated dynamically on demand using an initial state $s_0$ and a successor function $S(s) \to \{(a, s')\}$. A 2D grid maze is a classic implicit graph where states are coordinate tuples `(r, c)` and transitions are computed on-the-fly.

#### Valid Action Generation & Boundary Safety
From any state $(r, c)$, there are 4 candidate orthogonal moves:
$$\text{Up: } (r-1, c), \quad \text{Down: } (r+1, c), \quad \text{Left: } (r, c-1), \quad \text{Right: } (r, c+1)$$
A move is valid if and only if:
1. **Boundary Condition**: $0 \le \text{next\_r} < N$ and $0 \le \text{next\_c} < M$.
2. **Wall Collision Avoidance**: $\text{grid}[\text{next\_r}][\text{next\_c}] == 0$.

#### Manhattan Distance Relation (Admissible Heuristic)
The Manhattan distance ($L_1$ norm) between two grid points $(r_1, c_1)$ and $(r_2, c_2)$ is:
$$D_{\text{Manhattan}} = |r_1 - r_2| + |c_1 - c_2|$$
- In an **obstacle-free grid**, the BFS shortest path length is strictly equal to the Manhattan distance.
- When walls exist, detours are required, so $\text{BFS Steps} \ge D_{\text{Manhattan}}$. Thus, Manhattan distance is an **admissible heuristic** (never overestimates the true cost), forming the theoretical cornerstone for A* search.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in module path
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.bfs_maze_solver import (
    DEFAULT_SAMPLE_MAZE,
    NON_SQUARE_RECTANGULAR_MAZE,
    SERPENTINE_MAZE,
    OPEN_GRID_4X4_MAZE,
    SINGLE_CELL_MAZE,
    UNSOLVABLE_BLOCKED_MAZE,
    solve_maze_bfs,
    BFSMazeSolver,
    get_successors,
    calculate_manhattan_distance,
    visualize_maze_solution,
    get_maze_solver_viva_reference,
)

print("✓ Successfully imported BFS 2D Maze Solver modules!")

### 2. Solving the Standard Laboratory Sample Maze (Question 15)

**Sample Maze Input (3x3)**:
```python
grid = [
    [0, 1, 0],
    [0, 0, 0],
    [1, 1, 0]
]
```
- Start Coordinate: `(0, 0)` (Top-Left)
- Goal Coordinate: `(2, 2)` (Bottom-Right)
- Expected Output: `Shortest steps: 4`
- Optional Challenge: `Moves: "D, R, R, D"`

In [ ]:
sample_grid = [
    [0, 1, 0],
    [0, 0, 0],
    [1, 1, 0],
]

result = solve_maze_bfs(sample_grid, record_trace=True)

print("=== MAZE ASCII SOLUTION ===")
print(result.render_ascii_solution())

print("\n=== LABORATORY VERIFICATION ===")
print(f"Shortest steps: {result.steps}")
print(f"Optional Challenge Moves: \"{result.actions_str}\"")
print(f"Path Coordinates: {result.path}")
print(f"Manhattan Lower Bound: {result.manhattan_distance}")

### 3. Step-by-Step FIFO Queue & Action Execution Trace

BFS explores level-by-level using a FIFO queue (`collections.deque`). Below, we inspect each state expansion, the action taken to reach it, and the dynamic successors generated.

In [ ]:
print(f"{'Step':<5} | {'Popped (r,c)':<14} | {'Action':<8} | {'Dist':<5} | {'Frontier (FIFO Queue)':<22} | {'Generated Successors'}")
print("-" * 80)
for rec in result.trace:
    act = rec.action_taken if rec.action_taken else "START"
    frontier_repr = str(rec.frontier_snapshot)
    succ_repr = ", ".join(f"{a}:{c}" for a, c in rec.successors_generated)
    print(f"{rec.step_number:<5} | {str(rec.current_coord):<14} | {act:<8} | {rec.distance:<5} | {frontier_repr:<22} | {succ_repr}")

### 4. Non-Square Rectangular Grids ($N \ne M$)

The problem specification explicitly requires supporting non-square grids.
Below we demonstrate search on a 3-row by 5-column grid ($3 \times 5$).

In [ ]:
non_square_grid = [
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
]

rect_result = solve_maze_bfs(non_square_grid)
print(f"Dimensions: {len(non_square_grid)} rows x {len(non_square_grid[0])} cols")
print(rect_result.render_ascii_solution())
print(f"\nShortest steps: {rect_result.steps}")
print(f"Moves: \"{rect_result.actions_str}\"")

### 5. Manhattan Distance Relation: Obstacle-Free vs. Detours

Let us empirically prove the relationship between Manhattan distance and BFS shortest steps:
1. **Obstacle-Free Grid**: BFS steps strictly equal Manhattan distance ($|N-1| + |M-1|$).
2. **Serpentine Corridor Maze**: Walls force detours, causing BFS steps to strictly exceed Manhattan distance.

In [ ]:
# 1. Obstacle-Free Grid (4x4)
open_res = solve_maze_bfs(OPEN_GRID_4X4_MAZE)
print("--- Open Obstacle-Free Grid (4x4) ---")
print(open_res.render_ascii_solution())
print(f"BFS Shortest Steps: {open_res.steps} == Manhattan Distance: {open_res.manhattan_distance} (Equality holds!)")

# 2. Winding Serpentine Maze (5x5)
serp_res = solve_maze_bfs(SERPENTINE_MAZE)
print("\n--- Winding Serpentine Corridor Maze (5x5) ---")
print(serp_res.render_ascii_solution())
print(f"BFS Shortest Steps: {serp_res.steps} > Manhattan Distance: {serp_res.manhattan_distance} (Detour penalty = {serp_res.steps - serp_res.manhattan_distance} steps)")

### 6. Edge Cases: Unsolvable Mazes & Single-Cell Grids

- **Completely Blocked Maze**: The goal is sealed off by walls. The BFS queue empties without reaching the goal, returning `steps = -1` and `found = False`.
- **Single-Cell Grid**: Start is already goal `(0, 0) == (0, 0)`, returning `steps = 0` and `actions = []`.

In [ ]:
# Completely Blocked Maze
blocked_res = solve_maze_bfs(UNSOLVABLE_BLOCKED_MAZE)
print("Blocked Maze Found:", blocked_res.found)
print("Blocked Maze Steps:", blocked_res.steps)

# Single Cell Grid
single_res = solve_maze_bfs(SINGLE_CELL_MAZE)
print("Single Cell Steps:", single_res.steps)
print("Single Cell Moves:", repr(single_res.actions_str))

### 7. Graphical Visualization (Matplotlib)

Below, we render and save a visual diagram of the sample maze solution.

In [ ]:
diagram_path = Path("reports/bfs_maze_solution.png")
saved_img = visualize_maze_solution(sample_grid, result=result, output_path=diagram_path, show=False)
print(f"✓ Generated and saved visualization diagram to: {saved_img}")

### 8. Viva Voce Reference & Theory Guide

Core questions frequently asked during university lab examinations regarding BFS and Grid Mazes.

In [ ]:
viva = get_maze_solver_viva_reference()
for i, item in enumerate(viva["core_topics"], 1):
    print(f"Topic {i}: {item['topic']}")
    print(f"Q: {item['question']}")
    print(f"A: {item['answer']}\n")